# Mock Interview Auto-Grading Demo

This notebook shows the evaluation engine working through the real local API.

It will:
- create or reuse a local student account
- start interview sessions
- request an opening question
- submit sample answers to `/evaluate_response`
- print the returned scorecards and compare a weaker answer with a stronger one


## Before You Run It

1. Start the app with `npm run dev`.
2. If the dev server moved to a different port, update `BASE_URL` in the next cell.
3. This notebook uses only Python standard-library HTTP helpers, so it does not need `requests`.


In [ ]:
import http.cookiejar
import json
import pprint
import urllib.error
import urllib.request

BASE_URL = "http://127.0.0.1:3000"
EMAIL = "notebook.demo@example.com"
PASSWORD = "demo-pass-123"
DISPLAY_NAME = "Notebook Demo"

cookies = http.cookiejar.CookieJar()
opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(cookies))
pp = pprint.PrettyPrinter(width=100, sort_dicts=False)


def api_request(path, payload=None, method="POST"):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(f"{BASE_URL}{path}", data=data, method=method)
    if payload is not None:
        request.add_header("Content-Type", "application/json")

    try:
        with opener.open(request) as response:
            body = response.read().decode("utf-8")
            parsed = json.loads(body) if body else None
            return response.status, parsed
    except urllib.error.HTTPError as error:
        body = error.read().decode("utf-8")
        try:
            parsed = json.loads(body) if body else {}
        except json.JSONDecodeError:
            parsed = {"raw": body}
        message = parsed.get("error") or parsed.get("message") or str(parsed)
        raise RuntimeError(f"{error.code} {message}") from error


def signup_or_login(email=EMAIL, password=PASSWORD, display_name=DISPLAY_NAME):
    signup_payload = {
        "email": email,
        "password": password,
        "display_name": display_name,
        "account_type": "student"
    }

    try:
        _, payload = api_request("/auth/signup", signup_payload)
        return payload["user"], "signup"
    except RuntimeError as error:
        if "already exists" not in str(error).lower():
            raise
        _, payload = api_request(
            "/auth/login",
            {"email": email, "password": password}
        )
        return payload["user"], "login"


def start_session(mode, target_role, focus_area, self_critique_enabled=True):
    _, payload = api_request(
        "/start_session",
        {
            "target_role": target_role,
            "mode": mode,
            "focus_area": focus_area,
            "confidence_self_rating": 3,
            "question_limit": 5,
            "question_time_limit_seconds": None,
            "personalization_enabled": True,
            "self_critique_enabled": self_critique_enabled,
            "notes": "Notebook evaluation demo",
            "resume_text": None
        }
    )
    return payload


def ask_opening_question(session_id):
    _, payload = api_request(
        "/ask_question",
        {"session_id": session_id, "latest_answer": None}
    )
    return payload


def evaluate_answer(session_id, question_text, answer_text, target_role, mode, self_critique_enabled=True):
    _, payload = api_request(
        "/evaluate_response",
        {
            "session_id": session_id,
            "question_text": question_text,
            "answer_text": answer_text,
            "target_role": target_role,
            "mode": mode,
            "self_critique_enabled": self_critique_enabled
        }
    )
    return payload


def summarize_result(label, question_payload, evaluation_payload):
    scorecard = evaluation_payload["scorecard"]
    return {
        "label": label,
        "question": question_payload["question_text"],
        "rubric_id": scorecard["rubric_id"],
        "rubric_match_type": scorecard["rubric_match_type"],
        "overall_score": scorecard["overall_score"],
        "dimension_scores": scorecard["dimension_scores"],
        "strengths": scorecard["strengths"],
        "weak_skills": scorecard["weak_skills"],
        "overall_summary": scorecard["overall_summary"],
        "actionable_feedback": scorecard["actionable_feedback"],
        "growth_tips": scorecard["growth_tips"],
        "self_critique_output": scorecard.get("self_critique_output"),
        "rubric_coverage": scorecard["rubric_coverage"]
    }


def run_demo_answer(label, answer_text, mode="behavioral", target_role="Product Manager Intern", focus_area="leadership"):
    session = start_session(mode=mode, target_role=target_role, focus_area=focus_area)
    question = ask_opening_question(session["session_id"])
    evaluation = evaluate_answer(
        session_id=session["session_id"],
        question_text=question["question_text"],
        answer_text=answer_text,
        target_role=target_role,
        mode=mode,
        self_critique_enabled=True
    )
    return summarize_result(label, question, evaluation)


In [ ]:
user, auth_mode = signup_or_login()
print(f"Authenticated via: {auth_mode}")
pp.pprint(user)


In [ ]:
weak_answer = (
    "I tried to help the team. We had some issues and I communicated with people. "
    "It mostly worked out."
)

strong_answer = (
    "Situation: our launch was slipping because design, engineering, and ops each had different priorities. "
    "Task: as the student PM intern, I needed to align the team on one plan without formal authority. "
    "Action: I rebuilt the milestone plan around one shared launch metric, ran a short decision meeting with all leads, "
    "documented tradeoffs, and reassigned one lower-priority scope item so engineering could finish the highest-impact flow first. "
    "Result: we shipped on time, activation improved by 14%, and the team reused the same alignment format for later launches."
)

weak_result = run_demo_answer("Weak behavioral answer", weak_answer)
strong_result = run_demo_answer("Strong behavioral answer", strong_answer)


In [ ]:
print("Weak answer result")
pp.pprint(weak_result)

print("\nStrong answer result")
pp.pprint(strong_result)


In [ ]:
comparison = [
    {
        "label": weak_result["label"],
        "overall_score": weak_result["overall_score"],
        "weak_skills": weak_result["weak_skills"],
        "top_growth_tip": weak_result["growth_tips"][0] if weak_result["growth_tips"] else None
    },
    {
        "label": strong_result["label"],
        "overall_score": strong_result["overall_score"],
        "weak_skills": strong_result["weak_skills"],
        "top_growth_tip": strong_result["growth_tips"][0] if strong_result["growth_tips"] else None
    }
]

pp.pprint(comparison)


## Next Variations

To expand the demo, change `mode`, `target_role`, and `focus_area` inside `run_demo_answer(...)`.

Good follow-ups:
- swap in a technical answer and compare tradeoff-heavy vs vague reasoning
- swap in a case answer and compare structured recommendation vs unstructured brainstorming
- turn `self_critique_enabled=False` in `evaluate_answer(...)` to show the difference in returned scorecard fields
